# Flickr30K Captioning Evaluation — baseline vs AttnRes (3 seeds)

Evaluates the frozen-SigLIP → linear connector → GPT decoder VLM
(`SiglipAttnResCaptioner`) on the Flickr30K captioning split, comparing the
`baseline` and `attnres` decoder variants across seeds **123 / 456 / 789**.

**Pipeline stages (one section each):**
0. **W&B → Drive sync** — pull all project runs/artifacts into `MyDrive/…/wandb_sync` (run first if checkpoints are missing)
1. Dependency setup (`pycocoevalcap`, Java for METEOR/SPICE)
2. Mount Google Drive
3. Repo discovery + imports (reuses the existing codebase)
4. Config
5. `Evaluator` interface + `CaptioningEvaluator`
6. Checkpoint discovery (Drive `checkpoints/` + `wandb_sync/checkpoints/`, optional per-run W&B fallback)
7. Model loading + per-seed eval data
8. Generation + on-disk caching
9. Scoring + aggregation (mean ± std)
10. Results table (CIDEr first)

**Reuses existing logic** — checkpoints are loaded exactly like the training
script (`torch.load(...)[\"model\"]` into a freshly built `SiglipAttnResCaptioner`),
and the eval split is rebuilt with `src.vlm.flickr30k.load_flickr30k_examples`
using the **same seed and `max_examples`** so each checkpoint is scored on the
held-out images it never trained on (the 90/10 split is seed-dependent).

> Generation strategy: **greedy**. References: **all human captions per val image**.
> Designed to run on Colab (T4) with checkpoints on Google Drive.

## 0. W&B → Drive sync (run first if checkpoints are missing)

Mounts Google Drive, checks VM disk with `df`, then runs `scripts/migrate_wandb_to_drive.py` to download **all finished W&B runs** (checkpoints, histories, artifacts) into:

`MyDrive/AttnResGPT-next-scale-artifacts/wandb_sync/`

Set `RUN_WANDB_SYNC = False` to skip if you already synced. Requires `WANDB_API_KEY` (set below or in repo `.env`).

In [1]:


# --- replace me ---
import os
import shutil
from pathlib import Path

import wandb
from google.colab import drive

# --- replace / rotate if this was committed anywhere ---
os.environ["WANDB_API_KEY"] = "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"

drive.mount("/content/drive")

ENTITY = "atin5551-uc-davis"
PROJECT = "attnres-next-scale"
RUN_NAME = "vlm_baseline_flickr30k_b8_seed123"
CHECKPOINT_INTERVAL = 500
FINAL_STEP = 6750

DEST_DIR = Path(f"/content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/{RUN_NAME}")
TMP_DIR = Path("/tmp/wandb_ckpt_pull")
DEST_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# 500, 1000, ..., 6500, plus 6750
steps = list(range(CHECKPOINT_INTERVAL, FINAL_STEP, CHECKPOINT_INTERVAL))
if FINAL_STEP not in steps:
    steps.append(FINAL_STEP)

wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
api = wandb.Api()

ok, missing = [], []

for step in steps:
    dest_file = DEST_DIR / f"step_{step:07d}.pt"
    if dest_file.exists():
        print(f"SKIP (exists): {dest_file.name}")
        ok.append(step)
        continue

    artifact_ref = f"{ENTITY}/{PROJECT}/{RUN_NAME}_step_{step:07d}:latest"
    print(f"Fetching {artifact_ref} ...")
    try:
        artifact = api.artifact(artifact_ref)
        step_tmp = TMP_DIR / f"step_{step:07d}"
        step_tmp.mkdir(parents=True, exist_ok=True)
        artifact_dir = Path(artifact.download(root=str(step_tmp)))

        matches = sorted(artifact_dir.rglob(f"step_{step:07d}.pt"))
        if not matches:
            matches = sorted(artifact_dir.rglob("step_*.pt"))
        if not matches:
            raise FileNotFoundError(f"no .pt in {artifact_dir}")

        shutil.copy2(matches[0], dest_file)
        mb = dest_file.stat().st_size / 1e6
        print(f"  saved {dest_file.name} ({mb:.1f} MB)")
        ok.append(step)
    except Exception as exc:
        print(f"  MISSING: step {step} ({exc})")
        missing.append(step)

print("\nDone.")
print("Saved steps:", ok)
print("Missing steps:", missing)
print("Drive folder:", DEST_DIR)
print("Files:", sorted(p.name for p in DEST_DIR.glob("step_*.pt")))

Mounted at /content/drive


KeyboardInterrupt: 

In [13]:
from pathlib import Path
base = Path("/content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints")
for name in [
    "vlm_attnres_flickr30k_b8_seed456",
    "vlm_baseline_flickr30k_b8_seed789",
    "vlm_attnres_flickr30k_b8_seed789",
]:
    p = base / name / "step_0006750.pt"
    print(name, "OK" if p.exists() else "MISSING")

vlm_attnres_flickr30k_b8_seed456 OK
vlm_baseline_flickr30k_b8_seed789 OK
vlm_attnres_flickr30k_b8_seed789 OK


In [2]:
import hashlib
import json
import os
import shutil
from pathlib import Path

import wandb
from google.colab import drive

# --- config ---
os.environ["WANDB_API_KEY"] = "YOUR_KEY_HERE"
ENTITY = "atin5551-uc-davis"
PROJECT = "attnres-next-scale"
DEST = Path("/content/drive/MyDrive/AttnResGPT-next-scale-artifacts/wandb_sync_exact")
TMP = Path("/tmp/wandb_exact_dl")
MANIFEST = DEST / "sync_manifest.json"
STATE_FILTER = None  # None = all runs; or "finished"

drive.mount("/content/drive")
DEST.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

def file_md5(p: Path, chunk=1 << 20) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

def load_manifest():
    return json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {"artifacts": {}, "runs": {}}

def save_manifest(m):
    MANIFEST.write_text(json.dumps(m, indent=2))

wandb.login(relogin=True)
api = wandb.Api(timeout=120)
runs = list(api.runs(f"{ENTITY}/{PROJECT}", per_page=200))
if STATE_FILTER:
    runs = [r for r in runs if r.state == STATE_FILTER]

manifest = load_manifest()
total_art, ok_art, fail_art = 0, 0, 0

for run in runs:
    run_key = f"{run.id}:{run.name}"
    run_dir = DEST / "runs" / run.name
    run_dir.mkdir(parents=True, exist_ok=True)

    # --- run metadata (not artifact storage) ---
    if run_key not in manifest["runs"]:
        summary = {k: v for k, v in dict(run.summary).items() if not str(k).startswith("_")}
        config = {k: v for k, v in dict(run.config).items() if not str(k).startswith("_")}
        (run_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
        (run_dir / "config.json").write_text(json.dumps(config, indent=2, default=str))
        meta = {"id": run.id, "name": run.name, "state": run.state, "url": run.url}
        (run_dir / "meta.json").write_text(json.dumps(meta, indent=2))
        try:
            hist = run.history(samples=10_000_000)
            if len(hist):
                hist.to_parquet(run_dir / "history.parquet", index=False)
        except Exception as e:
            (run_dir / "history.error.txt").write_text(str(e))
        manifest["runs"][run_key] = "metadata_exported"
        save_manifest(manifest)

    # --- every logged artifact, every file in manifest ---
    for art in run.logged_artifacts():
        total_art += 1
        art_key = f"{art.name}@{art.version}"
        rel_root = Path("artifacts") / run.name / art.name.replace(":", "_")
        local_root = DEST / rel_root

        if manifest["artifacts"].get(art_key, {}).get("status") == "verified":
            ok_art += 1
            continue

        print(f"\n[{run.name}] {art.name}")
        try:
            dl_dir = TMP / art.name.replace(":", "_")
            if dl_dir.exists():
                shutil.rmtree(dl_dir)
            dl_dir.mkdir(parents=True)
            art.download(root=str(dl_dir))

            entries = {}
            for src in dl_dir.rglob("*"):
                if not src.is_file():
                    continue
                rel = src.relative_to(dl_dir)
                dst = local_root / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                entries[str(rel)] = {
                    "size_bytes": dst.stat().st_size,
                    "md5": file_md5(dst),
                }

            wandb_entries = {
                k: {"digest": v.digest, "size": v.size}
                for k, v in art.manifest.entries.items()
            }
            manifest["artifacts"][art_key] = {
                "run": run.name,
                "local_root": str(rel_root),
                "wandb_entries": wandb_entries,
                "local_files": entries,
                "status": "verified",
            }
            ok_art += 1
            save_manifest(manifest)
            shutil.rmtree(dl_dir, ignore_errors=True)
        except Exception as exc:
            manifest["artifacts"][art_key] = {
                "run": run.name, "status": "failed", "error": str(exc)
            }
            fail_art += 1
            save_manifest(manifest)
            print("  FAILED:", exc)

# --- summary ---
failed = [k for k, v in manifest["artifacts"].items() if v.get("status") != "verified"]
print("\n=== SYNC SUMMARY ===")
print("Runs:", len(runs))
print("Artifacts total:", total_art)    
print("Artifacts verified:", ok_art)
print("Artifacts failed:", fail_art)
print("Manifest:", MANIFEST)
print("SAFE TO DELETE W&B STORAGE:", len(failed) == 0 and fail_art == 0)
if failed:
    print("Still missing/failed:", failed[:20], "..." if len(failed) > 20 else "")

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc



[tinystories_small_baseline_ctx128_steps2_seed1337] run-tinystories_small_baseline_ctx128_steps2_seed1337-history:v0


wandb:   1 of 1 files downloaded.  


...


[vlm_attnres_flickr30k_b8_seed456] vlm_attnres_flickr30k_b8_seed456_step_0001000:v0


wandb: Downloading large artifact 'vlm_attnres_flickr30k_b8_seed456_step_0001000:v0', 1408.77MB. 5 files...
wandb:   5 of 5 files downloaded.  
Done. 00:00:29.4 (47.8MB/s)



[vlm_attnres_flickr30k_b8_seed456] vlm_attnres_flickr30k_b8_seed456_step_0001500:v0


wandb: Downloading large artifact 'vlm_attnres_flickr30k_b8_seed456_step_0001500:v0', 1408.77MB. 5 files...
wandb:   5 of 5 files downloaded.  
Done. 00:00:38.5 (36.6MB/s)


OSError: [Errno 28] No space left on device

## 1. Dependency setup

Installs `pycocoevalcap` (CIDEr, BLEU-4, METEOR, ROUGE-L, SPICE). METEOR, SPICE
and the PTB tokenizer need a JVM — on Colab we try to install one. If Java is
unavailable we fall back to a simple tokenizer and skip METEOR/SPICE.

In [2]:
import shutil
import subprocess
import sys


def _pip_install(*pkgs: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


try:
    import pycocoevalcap  # noqa: F401
except ImportError:
    _pip_install("pycocoevalcap")

# METEOR / SPICE / PTBTokenizer require a JVM. Best-effort install on Colab.
if shutil.which("java") is None:
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "default-jre"], check=True)
    except Exception as exc:  # noqa: BLE001
        print(f"Could not auto-install Java ({exc}); METEOR/SPICE will be skipped.")

# Pure-Python scorers are always available; import once for later cells.
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

JAVA_OK = shutil.which("java") is not None
print("pycocoevalcap OK | java available =", JAVA_OK)

pycocoevalcap OK | java available = True


## 2. Mount Google Drive (required)

Seed checkpoints live at `MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/`.
Section **0** can also populate `…/wandb_sync/checkpoints/` from W&B (eval searches both).
The Colab login and Drive account **can differ** — in the Google popup click
**Use another account** to pick the account that owns the checkpoints.
When Colab asks for Drive permissions, enable **all** checkboxes (read-only often fails).

In [3]:
from pathlib import Path

DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"
DRIVE_MOUNT = Path("/content/drive")
DRIVE_CHECKPOINTS = None


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if not _in_colab():
    raise RuntimeError("Run on Google Colab — VLM seed checkpoints are stored on Google Drive.")

from google.colab import auth, drive

print(
    "Mount the Google account that owns the checkpoint folder.\n"
    "Tip: click 'Use another account' if it defaults to your Colab login.\n"
    "Tip: enable ALL Drive permission boxes in the popup."
)

for attempt, force in enumerate([False, True], start=1):
    try:
        auth.authenticate_user()
        drive.mount(str(DRIVE_MOUNT), force_remount=force)
        DRIVE_CHECKPOINTS = DRIVE_MOUNT / "MyDrive" / DRIVE_ARTIFACT_FOLDER / "checkpoints"
        print("Drive mounted:", DRIVE_MOUNT)
        print("Checkpoint dir:", DRIVE_CHECKPOINTS, "| exists =", DRIVE_CHECKPOINTS.exists())
        break
    except Exception as exc:
        if attempt == 2:
            raise RuntimeError(
                "Google Drive mount failed — cannot load seed checkpoints.\n"
                "Try: Runtime > Disconnect and delete runtime, allow popups for colab.research.google.com,\n"
                "enable ALL Drive permissions, sign into the account with the checkpoints.\n"
                "Known Colab issue: https://github.com/googlecolab/colabtools/issues/5944"
            ) from exc
        print(f"Mount attempt {attempt} failed: {exc}\nRetrying with force_remount=True ...")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: /content/drive
Checkpoint dir: /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints | exists = True


## 3. Repo setup + imports

Clone/find the repo at `/content/AttnResGPT-next-scale` (no Drive filesystem scan), install deps, import from `src`. Requires section 2.


In [4]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


REPO_TARGET = Path("/content/AttnResGPT-next-scale")


def find_repo() -> Path | None:
    """Fast lookup only — never rglob Drive (that scans your entire MyDrive)."""
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    return None


if DRIVE_CHECKPOINTS is None:
    raise RuntimeError("Run the Google Drive mount cell (section 2) first.")

print("Looking for repo at fixed paths (no Drive scan) ...")
REPO = find_repo()
if REPO is None:
    print(f"Cloning into {REPO_TARGET} ...")
    if REPO_TARGET.exists():
        subprocess.run(["rm", "-rf", str(REPO_TARGET)], check=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    REPO = REPO_TARGET.resolve()
else:
    print(f"Found repo at {REPO}")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Installing requirements (may take a few minutes) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["git", "fetch", "--quiet"], check=False)
subprocess.run(["git", "pull", "--ff-only"], check=False)
print("repo root:", REPO)

import torch
from torch.utils.data import DataLoader, Dataset

from src.data.tokenizer import build_tokenizer
from src.models.vlm_attnres import SiglipAttnResCaptioner
from src.utils.config import load_config
from src.vlm.flickr30k import _row_captions, _row_image, load_flickr30k_examples

print("codebase imports OK")


Looking for repo at fixed paths (no Drive scan) ...
Cloning into /content/AttnResGPT-next-scale ...
Installing requirements (may take a few minutes) ...
repo root: /content/AttnResGPT-next-scale
codebase imports OK


## 4. Config

All knobs in one place. `MAX_EXAMPLES`, `DATASET_NAME`, `TOKENIZER_NAME`,
`VISION_MODEL` and `DECODER_CONFIG` must match training so the seed-dependent
90/10 split is reproduced exactly. `VLM_BATCH_SIZE` only affects the checkpoint
folder name (`vlm_{variant}_flickr30k_b{batch}_seed{seed}`).

In [5]:
SEEDS = [123, 456, 789]
VARIANTS = ["baseline", "attnres"]

# --- must match training (reproduces the split + rebuilds the model) ---
VISION_MODEL = "google/siglip-base-patch16-224"
DECODER_CONFIG = "configs/large_ctx512_3000.yaml"
TOKENIZER_NAME = "gpt2"
DATASET_NAME = "Mozilla/flickr30k-transformed-captions"
DATASET_SPLIT = "train"
MAX_EXAMPLES = 20000          # training default; drives the seed-dependent 90/10 split
DECODER_MAX_SEQ_LEN = 512
VLM_BATCH_SIZE = 8            # only affects checkpoint folder name
FINAL_CHECKPOINT = "step_0006750.pt"

# --- evaluation knobs ---
MAX_GEN_TOKENS = 40
GEN_BATCH_SIZE = 16
FORCE_REGEN = False           # True = ignore cached captions and re-generate

# --- optional W&B fallback for missing Drive checkpoints ---
ENABLE_WANDB_FALLBACK = False
WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale"

if DRIVE_CHECKPOINTS is None:
    raise RuntimeError("Run the Google Drive mount cell (section 2) first.")
DRIVE_WANDB_SYNC_CKPTS = (
    DRIVE_MOUNT / "MyDrive" / DRIVE_ARTIFACT_FOLDER / "wandb_sync" / "checkpoints"
)
CHECKPOINT_ROOTS = [DRIVE_CHECKPOINTS, DRIVE_WANDB_SYNC_CKPTS]
CACHE_DIR = REPO / "outputs" / "caption_eval"
DRIVE_CACHE_DIR = DRIVE_MOUNT / "MyDrive" / DRIVE_ARTIFACT_FOLDER / "outputs" / "caption_eval"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def caption_cache_filename(variant: str, seed: int) -> str:
    return f"{variant}_seed{seed}_captions.json"


def sync_cache_file_to_drive(path: Path) -> None:
    """Mirror a caption-eval artifact to Drive so it survives Colab disconnects."""
    if not path.exists():
        return
    import shutil

    dst = DRIVE_CACHE_DIR / path.name
    shutil.copy2(path, dst)
    print(f"  [drive] synced {path.name}")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("checkpoint roots:", [str(r) for r in CHECKPOINT_ROOTS])
print("local cache dir:", CACHE_DIR)
print("drive cache dir:", DRIVE_CACHE_DIR)


device: cuda
checkpoint roots: ['/content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints', '/content/drive/MyDrive/AttnResGPT-next-scale-artifacts/wandb_sync/checkpoints']
cache dir: /content/AttnResGPT-next-scale/outputs/caption_eval


## 5. Evaluator interface

`Evaluator` is an abstract strategy: `generate(model, dataloader)` produces
prediction records (each carrying its own references), `score(predictions)`
returns a metric dict, and `run` chains them. The generation/aggregation loop
below only ever calls these methods, so a future `VQAEvaluator` or
`RetrievalEvaluator` drops in as a new subclass with no pipeline changes.

`CaptioningEvaluator` implements greedy autoregressive decoding (the VLM has no
`.generate()`, so we roll our own) and scores with `pycocoevalcap`.

In [7]:
from abc import ABC, abstractmethod
from typing import Any


class Evaluator(ABC):
    """Abstract evaluation strategy. Subclass per task (captioning, VQA, ...)."""

    name: str = "base"

    @abstractmethod
    def generate(self, model: Any, dataloader: Any) -> list[dict]:
        """Run `model` over `dataloader`; return per-example prediction records."""

    @abstractmethod
    def score(self, predictions: list[dict]) -> dict[str, float]:
        """Score prediction records (they carry their own references)."""

    def run(self, model: Any, dataloader: Any) -> dict[str, float]:
        return self.score(self.generate(model, dataloader))


class CaptioningEvaluator(Evaluator):
    """Greedy caption generation + pycocoevalcap scoring (CIDEr/BLEU/METEOR/ROUGE/SPICE)."""

    name = "captioning"

    def __init__(
        self,
        tokenizer=None,
        device: "torch.device | None" = None,
        *,
        max_gen_tokens: int = 40,
        max_seq_len: int = 512,
        java_ok: bool = False,
    ) -> None:
        self.tokenizer = tokenizer
        self.device = device
        self.max_gen_tokens = max_gen_tokens
        self.max_seq_len = max_seq_len
        self.java_ok = java_ok
        # Token ids are only needed for generation (not scoring).
        self.bos_id = self.eos_id = None
        if tokenizer is not None:
            backend = tokenizer.backend
            pad = backend.pad_token_id if backend.pad_token_id is not None else 0
            bos = backend.bos_token_id
            if bos is None:
                bos = backend.eos_token_id
            if bos is None:
                bos = pad
            self.bos_id = int(bos)
            self.eos_id = int(backend.eos_token_id) if backend.eos_token_id is not None else int(pad)

    @torch.no_grad()
    def generate(self, model, dataloader) -> list[dict]:
        if self.tokenizer is None:
            raise ValueError("CaptioningEvaluator needs a tokenizer to generate.")
        model.eval()
        records: list[dict] = []
        for batch in dataloader:
            pixel_values = batch["pixel_values"].to(self.device)
            vision_hidden = model.encode_vision(pixel_values)
            batch_size = pixel_values.size(0)
            prefix_len = vision_hidden.size(1)
            input_ids = torch.full((batch_size, 1), self.bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)
            budget = max(1, min(self.max_gen_tokens, self.max_seq_len - prefix_len - 1))
            for _ in range(budget):
                output = model(vision_hidden=vision_hidden, input_ids=input_ids, return_aux=False)
                next_token = output["logits"][:, -1, :].argmax(dim=-1)
                next_token = torch.where(finished, torch.full_like(next_token, self.eos_id), next_token)
                input_ids = torch.cat([input_ids, next_token.unsqueeze(1)], dim=1)
                finished = finished | (next_token == self.eos_id)
                if bool(finished.all()):
                    break
            generated = input_ids[:, 1:].tolist()  # drop the seeded BOS
            for row, image_id, references in zip(generated, batch["image_ids"], batch["references"]):
                tokens: list[int] = []
                for token in row:
                    if token == self.eos_id:
                        break
                    tokens.append(int(token))
                caption = self.tokenizer.backend.decode(tokens).strip()
                records.append(
                    {"image_id": int(image_id), "prediction": caption, "references": list(references)}
                )
        return records

    def _tokenize(self, gts: dict, res: dict) -> tuple[dict, dict]:
        if self.java_ok:
            from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

            ptb = PTBTokenizer()
            return ptb.tokenize(gts), ptb.tokenize(res)

        def simple(table: dict) -> dict:
            return {key: [entry["caption"].lower().strip() for entry in value] for key, value in table.items()}

        return simple(gts), simple(res)

    def score(self, predictions: list[dict]) -> dict[str, float]:
        gts = {
            p["image_id"]: [{"caption": r} for r in p["references"] if isinstance(r, str) and r.strip()]
            for p in predictions
        }
        res = {
            p["image_id"]: [{"caption": p["prediction"] if p["prediction"].strip() else "."}]
            for p in predictions
        }
        valid = [k for k in gts if gts[k]]
        gts = {k: gts[k] for k in valid}
        res = {k: res[k] for k in valid}
        if not gts:
            return {}

        gts_tok, res_tok = self._tokenize(gts, res)
        scorers: list[tuple[Any, Any]] = [
            (Cider(), "CIDEr"),
            (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
            (Rouge(), "ROUGE_L"),
        ]
        if self.java_ok:
            from pycocoevalcap.meteor.meteor import Meteor

            scorers.append((Meteor(), "METEOR"))
            try:
                from pycocoevalcap.spice.spice import Spice

                scorers.append((Spice(), "SPICE"))
            except Exception as exc:  # noqa: BLE001
                print(f"  (SPICE unavailable: {exc})")

        metrics: dict[str, float] = {}
        for scorer, names in scorers:
            try:
                score, _ = scorer.compute_score(gts_tok, res_tok)
            except Exception as exc:  # noqa: BLE001
                print(f"  (scorer {names} failed: {exc})")
                continue
            if isinstance(names, list):
                for name, value in zip(names, score):
                    metrics[name] = float(value)
            else:
                metrics[names] = float(score)
        return metrics


print("Evaluator + CaptioningEvaluator defined")

Evaluator + CaptioningEvaluator defined


## 6. Checkpoint discovery

Looks for `vlm_{variant}_flickr30k_b{batch}_seed{seed}/step_0006750.pt` under each
checkpoint root (Drive first, then local), preferring the final step and falling
back to the latest available. Prints what it found so you can sanity-check before
generation. If `ENABLE_WANDB_FALLBACK` is set, missing checkpoints are pulled from
the W&B run artifact using the existing `src.analysis.attnres_wandb` helpers.

In [8]:
import re

CKPT_RE = re.compile(r"step_(\d+)\.pt$")


def run_folder_name(variant: str, seed: int) -> str:
    return f"vlm_{variant}_flickr30k_b{VLM_BATCH_SIZE}_seed{seed}"


def discover_checkpoint(variant: str, seed: int) -> "Path | None":
    folder = run_folder_name(variant, seed)
    for root in CHECKPOINT_ROOTS:
        run_dir = root / folder
        if not run_dir.exists():
            continue
        preferred = run_dir / FINAL_CHECKPOINT
        if preferred.exists():
            return preferred
        checkpoints = sorted(
            run_dir.glob("step_*.pt"),
            key=lambda p: int(CKPT_RE.search(p.name).group(1)),
        )
        if checkpoints:
            return checkpoints[-1]
    return None


def wandb_fallback(variant: str, seed: int, target_dir: Path) -> Path:
    """Best-effort download of a missing checkpoint from its W&B run artifact."""
    from src.analysis.attnres_wandb import (
        download_checkpoint_from_artifact,
        find_logged_checkpoint_artifact,
        login_wandb_from_env,
    )

    api = login_wandb_from_env()
    run_name = run_folder_name(variant, seed)
    runs = list(api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}", filters={"display_name": run_name}))
    if not runs:
        raise FileNotFoundError(f"No W&B run named {run_name}")
    run = runs[0]
    artifact = find_logged_checkpoint_artifact(api, run_path=f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run.id}")
    target_dir.mkdir(parents=True, exist_ok=True)
    path, _step = download_checkpoint_from_artifact(artifact, step=None, target_dir=target_dir)
    return path


discovered: dict[tuple[str, int], "Path | None"] = {}
print(f"{'variant':9} {'seed':>5}  checkpoint")
print("-" * 78)
for variant in VARIANTS:
    for seed in SEEDS:
        ckpt = discover_checkpoint(variant, seed)
        discovered[(variant, seed)] = ckpt
        print(f"{variant:9} {seed:>5}  {ckpt if ckpt else 'MISSING'}")

missing = [k for k, v in discovered.items() if v is None]
if missing:
    print("\nMissing:", missing, "| W&B fallback:", ENABLE_WANDB_FALLBACK)

variant    seed  checkpoint
------------------------------------------------------------------------------
baseline    123  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_baseline_flickr30k_b8_seed123/step_0006750.pt
baseline    456  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_baseline_flickr30k_b8_seed456/step_0006750.pt
baseline    789  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_baseline_flickr30k_b8_seed789/step_0006750.pt
attnres     123  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_attnres_flickr30k_b8_seed123/step_0006750.pt
attnres     456  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_attnres_flickr30k_b8_seed456/step_0006750.pt
attnres     789  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/vlm_attnres_flickr30k_b8_seed789/step_0006750.pt


## 7. Model loading + per-seed eval data

`load_vlm` rebuilds `SiglipAttnResCaptioner` with the same decoder config used in
training and loads `checkpoint["model"]` (identical to the training script's
resume path). `build_caption_eval_loader` rebuilds the **seed-specific** 90/10
split via `load_flickr30k_examples` and enforces an **image-level holdout**: it
keeps only val images whose captions never appear in train (dropping the single
boundary image), then gathers **all** human captions per image as references.

In [9]:
def build_vlm(variant: str):
    config = load_config(
        str(REPO / DECODER_CONFIG),
        overrides=[
            f"model.architecture={variant}",
            f"data.block_size={DECODER_MAX_SEQ_LEN}",
            f"model.max_seq_len={DECODER_MAX_SEQ_LEN}",
        ],
    )
    tokenizer = build_tokenizer(TOKENIZER_NAME)
    config.model.vocab_size = tokenizer.vocab_size
    model = SiglipAttnResCaptioner(vision_model_name=VISION_MODEL, decoder_config=config.model)
    return model, tokenizer


def load_vlm(variant: str, checkpoint_path: "str | Path"):
    model, tokenizer = build_vlm(variant)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE).eval()
    return model, tokenizer, checkpoint.get("step")


class CaptionGenDataset(Dataset):
    """Unique val images with all their reference captions."""

    def __init__(self, dataset, image_ids: list[int]) -> None:
        self.dataset = dataset
        self.image_ids = image_ids

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, index: int) -> dict:
        row_index = int(self.image_ids[index])
        row = self.dataset[row_index]
        return {"image_id": row_index, "image": _row_image(row), "references": _row_captions(row)}


class CaptionGenCollator:
    def __init__(self, processor) -> None:
        self.processor = processor

    def __call__(self, items: list[dict]) -> dict:
        images = [item["image"] for item in items]
        pixel_values = self.processor(images=images, return_tensors="pt")["pixel_values"]
        return {
            "pixel_values": pixel_values,
            "image_ids": [item["image_id"] for item in items],
            "references": [item["references"] for item in items],
        }


def build_caption_eval_loader(processor, seed: int, *, batch_size: int):
    dataset, train_records, val = load_flickr30k_examples(
        dataset_name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_examples=MAX_EXAMPLES,
        seed=seed,
    )
    # Enforce an image-level holdout: an image is a valid eval image only if NONE
    # of its captions appear in train. The underlying split is caption-level, so
    # this drops the single boundary image straddling the 90/10 cutoff, leaving a
    # leak-free set of images the model never saw during training.
    train_ids = {record.row_index for record in train_records}
    unique_ids: list[int] = []
    seen: set[int] = set()
    for record in val:
        row_index = record.row_index
        if row_index in train_ids or row_index in seen:
            continue
        seen.add(row_index)
        unique_ids.append(row_index)
    loader = DataLoader(
        CaptionGenDataset(dataset, unique_ids),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=CaptionGenCollator(processor),
    )
    return loader, len(unique_ids)


print("model loading + eval-data helpers defined")

model loading + eval-data helpers defined


## 8. Generation + caching

For each `(variant, seed)`: load the checkpoint, rebuild that seed's val loader,
greedily generate one caption per image, and cache `{image_id, prediction,
references}` to `outputs/caption_eval/{variant}_seed{seed}_captions.json` **and**
mirror each file to `MyDrive/…/outputs/caption_eval/` (survives Colab disconnects).
Loads from local cache first, then Drive. Set `FORCE_REGEN = True` to override.
Any failure is logged and the loop continues with the remaining checkpoints.

In [10]:
import gc
import json
import shutil
import traceback

all_predictions: dict[tuple[str, int], list[dict]] = {}
generation_log: list[tuple[str, int, str, int]] = []

for variant in VARIANTS:
    for seed in SEEDS:
        cache_path = CACHE_DIR / caption_cache_filename(variant, seed)
        drive_cache_path = DRIVE_CACHE_DIR / caption_cache_filename(variant, seed)
        try:
            if not FORCE_REGEN:
                if cache_path.exists():
                    predictions = json.loads(cache_path.read_text())
                    all_predictions[(variant, seed)] = predictions
                    generation_log.append((variant, seed, "cached", len(predictions)))
                    print(f"[cache] {variant} seed{seed}: {len(predictions)} captions (local)")
                    continue
                if drive_cache_path.exists():
                    shutil.copy2(drive_cache_path, cache_path)
                    predictions = json.loads(cache_path.read_text())
                    all_predictions[(variant, seed)] = predictions
                    generation_log.append((variant, seed, "drive_cache", len(predictions)))
                    print(f"[cache] {variant} seed{seed}: {len(predictions)} captions (drive -> local)")
                    continue

            checkpoint_path = discovered.get((variant, seed))
            if checkpoint_path is None and ENABLE_WANDB_FALLBACK:
                try:
                    checkpoint_path = wandb_fallback(
                        variant, seed, REPO / "checkpoints" / run_folder_name(variant, seed)
                    )
                    print(f"[wandb] pulled checkpoint for {variant} seed{seed}: {checkpoint_path}")
                except Exception as exc:  # noqa: BLE001
                    print(f"[wandb] fallback failed for {variant} seed{seed}: {exc}")

            if checkpoint_path is None:
                generation_log.append((variant, seed, "missing_checkpoint", 0))
                print(f"[skip] {variant} seed{seed}: no checkpoint")
                continue

            model, tokenizer, step = load_vlm(variant, checkpoint_path)
            evaluator = CaptioningEvaluator(
                tokenizer,
                DEVICE,
                max_gen_tokens=MAX_GEN_TOKENS,
                max_seq_len=DECODER_MAX_SEQ_LEN,
                java_ok=JAVA_OK,
            )
            loader, n_images = build_caption_eval_loader(model.processor, seed, batch_size=GEN_BATCH_SIZE)
            print(f"[gen] {variant} seed{seed} (step={step}): generating for {n_images} val images ...")
            predictions = evaluator.generate(model, loader)
            cache_path.write_text(json.dumps(predictions, indent=2))
            sync_cache_file_to_drive(cache_path)
            all_predictions[(variant, seed)] = predictions
            generation_log.append((variant, seed, "generated", len(predictions)))
            print(f"[gen] {variant} seed{seed}: {len(predictions)} captions -> {cache_path.name}")

            del model, evaluator, loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as exc:  # noqa: BLE001
            generation_log.append((variant, seed, f"error: {exc}", 0))
            print(f"[error] {variant} seed{seed}: {exc}")
            traceback.print_exc()

print("\nGeneration summary")
print("-" * 50)
for variant, seed, status, count in generation_log:
    print(f"  {variant:9} seed{seed}: {status} ({count})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

[gen] attnres seed123 (step=6750): generating for 2000 val images ...
[gen] attnres seed123: 2000 captions -> attnres_seed123_captions.json


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

[gen] attnres seed456 (step=6750): generating for 2000 val images ...
[gen] attnres seed456: 2000 captions -> attnres_seed456_captions.json


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

[gen] attnres seed789 (step=6750): generating for 2000 val images ...
[gen] attnres seed789: 2000 captions -> attnres_seed789_captions.json

Generation summary
--------------------------------------------------
  baseline  seed123: generated (2000)
  baseline  seed456: generated (2000)
  baseline  seed789: generated (2000)
  attnres   seed123: generated (2000)
  attnres   seed456: generated (2000)
  attnres   seed789: generated (2000)


## 9. Scoring + aggregation

Scores the cached captions per `(variant, seed)`, then aggregates across seeds:
mean ± sample std (ddof=1) for every metric. Scoring reads only from
`all_predictions`, so you can re-run this section without re-generating.

In [11]:
import numpy as np

# Reused for scoring only (no model/tokenizer needed).
_scorer = CaptioningEvaluator(java_ok=JAVA_OK)

per_seed_metrics: dict[tuple[str, int], dict[str, float]] = {}
for (variant, seed), predictions in all_predictions.items():
    try:
        metrics = _scorer.score(predictions)
        per_seed_metrics[(variant, seed)] = metrics
        cider = metrics.get("CIDEr")
        print(f"[score] {variant} seed{seed}: CIDEr={cider:.4f}" if cider is not None
              else f"[score] {variant} seed{seed}: {metrics}")
    except Exception as exc:  # noqa: BLE001
        print(f"[score error] {variant} seed{seed}: {exc}")


def aggregate(variant: str) -> tuple[dict[str, tuple[float, float]], int]:
    rows = [per_seed_metrics[(variant, s)] for s in SEEDS if (variant, s) in per_seed_metrics]
    if not rows:
        return {}, 0
    keys = set().union(*[set(r.keys()) for r in rows])
    aggregated: dict[str, tuple[float, float]] = {}
    for key in keys:
        values = [r[key] for r in rows if key in r]
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
        aggregated[key] = (mean, std)
    return aggregated, len(rows)


aggregated_metrics = {variant: aggregate(variant) for variant in VARIANTS}
for variant, (agg, n) in aggregated_metrics.items():
    print(f"\n{variant} (n={n} seeds)")
    for key in sorted(agg):
        mean, std = agg[key]
        print(f"  {key:10} {mean:.4f} ± {std:.4f}")

Progress: 384.5M / 384.5M (100.0%)
Extracting stanford-corenlp-3.6.0 ...
Done.
{'testlen': 52085, 'reflen': 32749, 'guess': [52085, 50085, 48085, 46085], 'correct': [13849, 5530, 2156, 959]}
ratio: 1.5904302421447498
  (scorer SPICE failed: Command '['java', '-jar', '-Xmx8G', 'spice-1.0.jar', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/tmp/tmpvplvecxd', '-cache', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/cache', '-out', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/tmp/tmppfe_7zls', '-subset', '-silent']' returned non-zero exit status 1.)
[score] baseline seed123: CIDEr=0.1893
{'testlen': 45006, 'reflen': 33292, 'guess': [45006, 43006, 41006, 39006], 'correct': [14115, 5600, 2148, 971]}
ratio: 1.3518563018142091
  (scorer SPICE failed: Command '['java', '-jar', '-Xmx8G', 'spice-1.0.jar', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/tmp/tmpj2fgqtyp', '-cache', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/

## 10. Results

Final comparison: rows = variants, columns = metrics (CIDEr first), cells =
mean ± std across seeds. Also writes the aggregated + per-seed numbers to
`outputs/caption_eval/results_summary.json` and mirrors it to Drive.

In [12]:
import pandas as pd

METRIC_ORDER = ["CIDEr", "Bleu_4", "METEOR", "ROUGE_L", "SPICE"]

present_metrics = [
    m for m in METRIC_ORDER
    if any(m in per_seed_metrics.get((v, s), {}) for v in VARIANTS for s in SEEDS)
]

table_rows: dict[str, dict[str, str]] = {}
seed_counts: dict[str, int] = {}
for variant in VARIANTS:
    agg, n = aggregated_metrics[variant]
    seed_counts[variant] = n
    table_rows[variant] = {
        metric: (f"{agg[metric][0]:.4f} ± {agg[metric][1]:.4f}" if metric in agg else "—")
        for metric in present_metrics
    }

comparison = pd.DataFrame(table_rows).T
comparison = comparison[present_metrics] if present_metrics else comparison
comparison.index.name = "variant"

print("Seeds aggregated per variant:", seed_counts)
print("Metric backend:", "PTBTokenizer + METEOR/SPICE" if JAVA_OK else "simple tokenizer (BLEU/ROUGE/CIDEr only)")

summary_payload = {
    "variants": VARIANTS,
    "seeds": SEEDS,
    "metrics": present_metrics,
    "seed_counts": seed_counts,
    "java_metrics_enabled": bool(JAVA_OK),
    "aggregated": {
        variant: {m: {"mean": agg[m][0], "std": agg[m][1]} for m in agg}
        for variant, (agg, _n) in aggregated_metrics.items()
    },
    "per_seed": {
        f"{variant}_seed{seed}": per_seed_metrics[(variant, seed)]
        for variant in VARIANTS
        for seed in SEEDS
        if (variant, seed) in per_seed_metrics
    },
}
summary_path = CACHE_DIR / "results_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2))
sync_cache_file_to_drive(summary_path)
print("wrote", summary_path)

comparison

Seeds aggregated per variant: {'baseline': 3, 'attnres': 3}
Metric backend: PTBTokenizer + METEOR/SPICE
wrote /content/AttnResGPT-next-scale/outputs/caption_eval/results_summary.json


,CIDEr,Bleu_4,METEOR,ROUGE_L
variant,,,,
baseline,0.3052 ± 0.1026,0.0822 ± 0.0087,0.1660 ± 0.0036,0.3231 ± 0.0190
attnres,0.1060 ± 0.0556,0.0685 ± 0.0033,0.1710 ± 0.0032,0.2978 ± 0.0092
